# Vision-Language Model Training Pipeline

This notebook demonstrates the training and evaluation of a Vision-Language Model (VLM) using Q-Former and small language models. The pipeline includes dataset preparation, Q-Former training, fine-tuning the language model, and evaluation.

In [2]:
import torch
from PIL import Image
from transformers import (
    AutoTokenizer,
    ViTModel,
    ViTImageProcessor,
)
from vlm_train.networks.lm_to_vlm import LM_2_VLM

# -------------------------
# Device
# -------------------------
device = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)

# -------------------------
# Config
# -------------------------
IMAGE_PATH = "test.png"
BASE_MODEL = "HuggingFaceTB/SmolLM-135M-Instruct"
CHECKPOINT_PATH = "models/vlm_peft/best"
QFORMER_PATH = "models/trained_qformer/best"

# -------------------------
# Load Tokenizer
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -------------------------
# Load VLM
# -------------------------
model = LM_2_VLM(
    model_name=BASE_MODEL,
    qformer_model_path=QFORMER_PATH,
    pad_token_id=tokenizer.pad_token_id,
)

model.load_checkpoint(CHECKPOINT_PATH)
model.to(device)
model.eval()

# -------------------------
# Load ViT (same as training)
# -------------------------
vit_name = "google/vit-base-patch16-224"
vit_processor = ViTImageProcessor.from_pretrained(vit_name)
vit_model = ViTModel.from_pretrained(vit_name).to(device)
vit_model.eval()

# -------------------------
# Load Image
# -------------------------
image = Image.open(IMAGE_PATH).convert("RGB")

inputs = vit_processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    vit_outputs = vit_model(**inputs)
    image_embeddings = vit_outputs.last_hidden_state  # (1, 197, 768)


trainable params: 19,537,920 || all params: 154,052,928 || trainable%: 12.6826
Loaded QFormer weights.
Loaded Adapter weights.
Loaded LoRA Adapter weights.


Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
# -------------------------
# Create Prompt
# -------------------------

prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "Answer the user's question truthfully"},
        {"role": "user", "content": "?"},
    ],
    return_tensors="pt",
).to(device)

# -------------------------
# Generate
# -------------------------
with torch.no_grad():
    output_ids = model.generate(
        img=image_embeddings,
        prefix_ids=prompt,
        max_new_tokens=100,
    )

# -------------------------
# Decode
# -------------------------
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("\n=== GENERATED OUTPUT ===\n")
print(output_text)


=== GENERATED OUTPUT ===

pink roses on a white background stock photos, images and royalty free photography. rose bouquet in the park stock photos, pictures & HD wallpaper flowers for phone and wallpapers 1600x850 stock photo is a high resolution image from . <PERSON>, <PERSON> Flowers, Blue Flower Baskets, Blue Roses, <PERSON>, <PERSON>, Rope Bouquets, Rope Flowers, Wedding Bouquets


In [2]:


# -------------------------
# Device
# -------------------------
device = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)

# -------------------------
# Config
# -------------------------
IMAGE_PATH = "test.png"
BASE_MODEL = "HuggingFaceTB/SmolLM-135M-Instruct"
CHECKPOINT_PATH = "models/vlm_peft/best"
QFORMER_PATH = "models/trained_qformer/best"

# -------------------------
# Load Tokenizer
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -------------------------
# Load VLM
# -------------------------
model = LM_2_VLM(
    model_name=BASE_MODEL,
    qformer_model_path=QFORMER_PATH,
    pad_token_id=tokenizer.pad_token_id,
)

model.load_checkpoint(CHECKPOINT_PATH)
model.to(device)
model.eval()

# -------------------------
# Load ViT (same as training)
# -------------------------
vit_name = "google/vit-base-patch16-224"
vit_processor = ViTImageProcessor.from_pretrained(vit_name)
vit_model = ViTModel.from_pretrained(vit_name).to(device)
vit_model.eval()

# -------------------------
# Load Image
# -------------------------
image = Image.open(IMAGE_PATH).convert("RGB")

inputs = vit_processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    vit_outputs = vit_model(**inputs)
    image_embeddings = vit_outputs.last_hidden_state  # (1, 197, 768)

# -------------------------
# Create Prompt
# -------------------------
prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "Answer the user's question truthfully"},
        {"role": "user", "content": "Describe this image in detail."},
    ],
    return_tensors="pt",
).to(device)

# -------------------------
# Generate
# -------------------------
with torch.no_grad():
    output_ids = model.generate(
        img=image_embeddings,
        prefix_ids=prompt,
        max_new_tokens=100,
    )

# -------------------------
# Decode
# -------------------------
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("\n=== GENERATED OUTPUT ===\n")
print(output_text)



'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /HuggingFaceTB/SmolLM-135M-Instruct/resolve/main/tokenizer_config.json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))"), '(Request ID: 8d504ac4-1fd0-4867-9edf-234ba8f8d909)')' thrown while requesting HEAD https://huggingface.co/HuggingFaceTB/SmolLM-135M-Instruct/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /HuggingFaceTB/SmolLM-135M-Instruct/resolve/main/tokenizer_config.json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))"), '(Request ID: 3151f98a-abcb-4b62-afc4-a84d4de0ad3d)')' thrown while requesting HEAD https://huggingface.co/Hugg

trainable params: 19,537,920 || all params: 154,052,928 || trainable%: 12.6826
Loaded QFormer weights.
Loaded Adapter weights.
Loaded LoRA Adapter weights.


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /google/vit-base-patch16-224/resolve/main/preprocessor_config.json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))"), '(Request ID: e83602b6-ef36-4edf-af17-88d3d0c6f2f3)')' thrown while requesting HEAD https://huggingface.co/google/vit-base-patch16-224/resolve/main/preprocessor_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /google/vit-base-patch16-224/resolve/main/preprocessor_config.json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))"), '(Request ID: 7aa6951b-770b-401d-bafe-521f30a3258c)')' thrown while requesting HEAD https://huggingface.co/google/vit-base-

SSLError: (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /google/vit-base-patch16-224/resolve/main/processor_config.json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))"), '(Request ID: e3caf338-4e7f-4984-ac4f-35d7cfff9620)')

## 1. Setup and Imports

Import necessary libraries and set up the environment, including PyTorch, Hugging Face Transformers, and project-specific modules.

In [ ]:
# Install dependencies
!pip install torch torchvision transformers img2dataset

# Import libraries
import torch
from transformers import AutoTokenizer, AutoModel

# Project-specific imports
from vlm_train.utils.filter_dataset import main as filter_dataset
from vlm_train.q_former_train import main as train_q_former
from vlm_train.lm_train import main as train_lm
from vlm_train.test_generation import main as generate_captions
from vlm_train.basic_inference import main as evaluate_metrics

## 2. Dataset Preparation

Use the `filter_dataset.py` script to download and preprocess the Conceptual Captions dataset. Optionally, use `img2dataset` to download and resize images.

In [ ]:
# Run the dataset preparation script
filter_dataset()

# Optionally, use img2dataset to download and resize images
!img2dataset --url_list dataset/conceptual-captions-200k.parquet \
             --input_format "parquet" \
             --url_col "url" \
             --caption_col "caption" \
             --output_folder dataset/cc_images \
             --processes_count 16 \
             --thread_count 64 \
             --image_size 224 \
             --resize_mode center_crop

## 3. Train Q-Former

Run the `q_former_train.py` script to train the Q-Former for vision-language alignment. Include key hyperparameters such as learning rate, batch size, and loss function.

In [ ]:
# Train the Q-Former
train_q_former() 

## 4. Fine-Tune Vision-Language Model

Run the `lm_train.py` script to fine-tune the language model with the trained Q-Former. Include hyperparameters like learning rate, LoRA configuration, and batch size.

In [ ]:
# Fine-tune the Vision-Language Model
train_lm()

## 5. Generate Captions

Use the `test_generation.py` script to generate captions for test images and save the results to a CSV file.

In [ ]:
# Generate captions for test images
generate_captions()

## 6. Evaluate Retrieval Metrics

Run the `basic_inference.py` script to compute image-to-text and text-to-image retrieval metrics. Visualize the similarity grid and display Recall@K metrics.

In [ ]:
# Evaluate retrieval metrics
evaluate_metrics()